In [92]:
from fastai.vision.all import *
# from sklearn.metrics import roc_curve, auc
# from fastai.metrics import * 

# from pathlib import Path
from sklearn.model_selection import   RepeatedKFold, GroupKFold, GroupShuffleSplit, StratifiedGroupKFold, LeaveOneGroupOut, LeavePGroupsOut
from sklearn.utils import resample
import sklearn.metrics as skm
from pathlib import Path
import numpy as np
from numpy import random
import shutil
import glob
import os
import pandas as pd
path = Path('/ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v6')

In [93]:
?GroupShuffleSplit

Init signature:
GroupShuffleSplit(
    n_splits=5,
    *,
    test_size=None,
    train_size=None,
    random_state=None,
)
Docstring:     
Shuffle-Group(s)-Out cross-validation iterator

Provides randomized train/test indices to split data according to a
third-party provided group. This group information can be used to encode
arbitrary domain specific stratifications of the samples as integers.

For instance the groups could be the year of collection of the samples
and thus allow for cross-validation against time-based splits.

The difference between LeavePGroupsOut and GroupShuffleSplit is that
the former generates splits using all subsets of size ``p`` unique groups,
whereas GroupShuffleSplit generates a user-determined number of random
test splits, each with a user-determined fraction of unique groups.

For example, a less computationally intensive alternative to
``LeavePGroupsOut(p=10)`` would be
``GroupShuffleSplit(test_size=10, n_splits=100)``.

Note: The parameters ``test_size`

In [5]:
# Create some regular expression queries:
use = 'tiles'
re_class= r"class_(.+)_x\d+_y\d+.jpg"
re_slide =r'(.+)_class_\S+_x\d+_y\d+.jpg$'
re_slide_class=r'(.+)_class_(.+)_x\d+_y\d+.jpg$'
slides=[]
slides_c=[]
all_c=[]
all_f=[]
for f in (path/use).ls():
    all_f.append(f.name)
    slide, c = re.findall(re_slide_class,f.name)[0]
    slides_c.append("%s_%s" % (slide,c))
    slides.append(slide)
    all_c.append(c)
print(np.unique(slides_c), len(np.unique(slides_c)))
posf = np.array([(f,c,s) for f,c,s in zip(all_f,all_c,slides) if c =='pos'])
negf = np.array([(f,c,s) for f,c,s in zip(all_f,all_c,slides) if c =='neg'])
all_dat=pd.DataFrame(all_f,columns=['fn'])
all_dat['slide']=slides
all_dat['class']=all_c
print(all_dat.shape)
u_slides_c=np.unique(slides_c)
temp = [item.split('_') for item in u_slides_c]
slide_df=pd.DataFrame(temp,columns=['slide','group'])
slide_df['class']=slide_df.loc[:,'group']=='pos'

['1007466_neg' '1007467_neg' '1007468_neg' '1007469_neg' '1007470_neg'
 '1007471_neg' '1007473_neg' '1007474_neg' '1007476_pos' '1007477_pos'
 '1007478_pos' '1007482_pos' '1007484_pos' '1007485_pos' '1007486_pos'
 '1007720_neg' '1007726_neg' '1007731_pos' '1007733_pos' '1007820_neg'
 '1007821_neg' '1007822_neg' '1007824_neg' '1007825_neg' '1007826_pos'
 '1007827_pos' '1007828_pos' '1007829_pos' '1007830_pos' '1007831_pos'
 '1007832_pos' '1007845_neg' '1007846_neg' '1007847_neg' '1007848_neg'] 35
(51813, 3)


In [7]:
#Example of Leave-One-Out cross-validation:
u_slides_c=np.unique(slides_c)
temp = [item.split('_') for item in u_slides_c]
slide_df=pd.DataFrame(temp,columns=['slide','group'])
slide_df['class']=slide_df.loc[:,'group']=='pos'
logo=LeaveOneGroupOut()
fold = 0 
    
for slide_train_idx, slide_test_idx in logo.split(slide_df.loc[:,'slide'],
                            slide_df.loc[:,'group'],                                         
                            groups=slide_df.loc[:,'slide']):
    print('\nFold %d' % fold)
    print(len(slide_train_idx),len(slide_test_idx))
    fold += 1


Fold 0
34 1

Fold 1
34 1

Fold 2
34 1

Fold 3
34 1

Fold 4
34 1

Fold 5
34 1

Fold 6
34 1

Fold 7
34 1

Fold 8
34 1

Fold 9
34 1

Fold 10
34 1

Fold 11
34 1

Fold 12
34 1

Fold 13
34 1

Fold 14
34 1

Fold 15
34 1

Fold 16
34 1

Fold 17
34 1

Fold 18
34 1

Fold 19
34 1

Fold 20
34 1

Fold 21
34 1

Fold 22
34 1

Fold 23
34 1

Fold 24
34 1

Fold 25
34 1

Fold 26
34 1

Fold 27
34 1

Fold 28
34 1

Fold 29
34 1

Fold 30
34 1

Fold 31
34 1

Fold 32
34 1

Fold 33
34 1

Fold 34
34 1


# Debug Leave-one-out jack-knife

In [8]:
#Debug leave-one-out jack-knife
seed=42
run_name='resnet18_jackknife'

balance_slides=[False,False]
balance_total_tiles=[True, False] #Train, Test
fold=0
model_path = '%s_model' % use
export_path=path.joinpath(model_path).joinpath(run_name+'/fold_models')
if export_path.exists()==False:
    os.makedirs(export_path)
csv_path=path.joinpath(model_path).joinpath(run_name+'/csv')
if csv_path.exists()==False:
    os.makedirs(csv_path)
    
logo=LeaveOneGroupOut()
for slide_train_idx, slide_test_idx in logo.split(slide_df.loc[:,'slide'],
                            slide_df.loc[:,'group'],                                         
                            groups=slide_df.loc[:,'slide']):
    print('\nFold %d' % fold)
    sets=[slide_train_idx,slide_test_idx]
    set_lab=['train','valid']
    df_list=[]
    n_train_test = [np.sum(slide_df.loc[slide_train_idx,'class']==True), #n pos
                    np.sum(slide_df.loc[slide_train_idx,'class']==False), #n neg
                    np.sum(slide_df.loc[slide_test_idx,'class']==True), #n pos
                    np.sum(slide_df.loc[slide_test_idx,'class']==False)] #n neg
    print('Train slides n=%d pos, %d neg, Test slides n=%d pos, %d neg' % tuple(n_train_test))
    for i,ds in enumerate(sets):   
        use_slides=list(slide_df.loc[ds,'slide'])
        use_tiles=all_dat.loc[:,'slide'].isin(use_slides) #Index of tiles from slide(s) in set (train / valid)
        #balance slides (?):
        #balance per-slide tile (?):
        
        bal_dat=pd.DataFrame(all_dat.loc[use_tiles,('fn','slide','class')]).reset_index(drop=True)
        
        if balance_total_tiles[i] == True:
            #Resample tiles to have equal number of class examples following previous balancing:        
            
            use_fn=bal_dat.loc[use_tiles,'fn']
            use_class=bal_dat.loc[use_tiles,'class']=='pos'
            npos=np.sum(use_class)
            nneg=len(use_class)-npos
            use_n = np.min([npos,nneg])
            print('Balancing %s tiles... %d neg %d pos, use %d' % (set_lab[i],nneg, npos,use_n))
            if npos > nneg:
                old_pos=np.argwhere(np.array(use_class)==True)
                neg=np.argwhere(np.array(use_class)==False)
                new=resample(old_pos,
                             replace=False,
                             n_samples=use_n,
                             random_state=seed)
                bal_tiles_idx=np.concatenate((neg,new))
            elif nneg > npos:
                old_neg=np.argwhere(np.array(use_class)==True)
                pos=np.argwhere(np.array(use_class)==False)
                new=resample(old_neg,
                             replace=False,
                             n_samples=use_n,
                             random_state=seed)
                bal_tiles_idx=np.concatenate((new,pos))
            temp=pd.DataFrame(bal_dat.loc[bal_tiles_idx[:,0],('fn','slide','class')]).reset_index(drop=True)
        else:
            use_class=bal_dat.loc[:,'class']=='pos'
            npos=np.sum(use_class)
            nneg=len(use_class)-npos
            print('Not balancing %s ... %d neg %d pos, use all.' % (set_lab[i],nneg, npos))
            temp=pd.DataFrame(bal_dat.loc[:,('fn','slide','class')]).reset_index(drop=True)
        temp['is_valid']=i
        if i == 0:
            out=temp
        else:
            out=pd.concat((out,temp),axis=0)
    
    full_path_fn=[path.joinpath(use).joinpath(fn) for fn in out['fn']]
    out['full_path']=full_path_fn
    print('Save to %s' % csv_path.joinpath('train_valid_fold_%d.csv' % fold))
    out.to_csv(csv_path.joinpath('train_valid_fold_%d.csv' % fold))
    fold+=1
    


Fold 0
Train slides n=16 pos, 18 neg, Test slides n=0 pos, 1 neg
Balancing train tiles... 15955 neg 34238 pos, use 15955
Not balancing valid ... 815 neg 0 pos, use all.
Save to /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v6/tiles_model/resnet18_jackknife/csv/train_valid_fold_0.csv

Fold 1
Train slides n=16 pos, 18 neg, Test slides n=0 pos, 1 neg
Balancing train tiles... 16086 neg 34306 pos, use 16086
Not balancing valid ... 714 neg 0 pos, use all.
Save to /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v6/tiles_model/resnet18_jackknife/csv/train_valid_fold_1.csv

Fold 2
Train slides n=16 pos, 18 neg, Test slides n=0 pos, 1 neg
Balancing train tiles... 14705 neg 33574 pos, use 14705
Not balancing valid ... 1794 neg 0 pos, use all.
Save to /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v6/tiles_model/resnet18_jackknife/csv/train_valid_fold_2.csv

Fold 3
Train slides n=16 pos, 18 neg, Test slides n=0 po

# Debug 10-fold valid

In [28]:
#Debug 10k-fold``
seed=10
run_name='resnet18_true_10kfold'
u_slides_c=np.unique(slides_c)
temp = [item.split('_') for item in u_slides_c]
slide_df=pd.DataFrame(temp,columns=['slide','group'])
slide_df['class']=slide_df.loc[:,'group']=='pos'
gkf=GroupKFold(n_splits=10)
gkf.get_n_splits(slide_df.loc[:,'slide'],
                            slide_df.loc[:,'group'],
                            groups=slide_df.loc[:,'slide'])
balance_slides=[False,False]
balance_total_tiles=[True, False] #Train, Test
fold=0
model_path = '%s_model' % use
export_path=path.joinpath(model_path).joinpath(run_name+'/fold_models')
if export_path.exists()==False:
    os.makedirs(export_path)
csv_path=path.joinpath(model_path).joinpath(run_name+'/csv')
if csv_path.exists()==False:
    os.makedirs(csv_path)
    
for slide_train_idx, slide_test_idx in gkf.split(slide_df.loc[:,'slide'],
                            slide_df.loc[:,'group'],                                         
                            groups=slide_df.loc[:,'slide']):
    print('\nFold %d' % fold)
    sets=[slide_train_idx,slide_test_idx]
    set_lab=['train','valid']
    df_list=[]
    n_train_test = [np.sum(slide_df.loc[slide_train_idx,'class']==True), #n pos
                    np.sum(slide_df.loc[slide_train_idx,'class']==False), #n neg
                    np.sum(slide_df.loc[slide_test_idx,'class']==True), #n pos
                    np.sum(slide_df.loc[slide_test_idx,'class']==False)] #n neg
    print('Train slides n=%d pos, %d neg, Test slides n=%d pos, %d neg' % tuple(n_train_test))
    for i,ds in enumerate(sets):   
        use_slides=list(slide_df.loc[ds,'slide'])
        use_tiles=all_dat.loc[:,'slide'].isin(use_slides) #Index of tiles from slide(s) in set (train / valid)
        #balance slides (?):
        #balance per-slide tile (?):
        
        bal_dat=pd.DataFrame(all_dat.loc[use_tiles,('fn','slide','class')]).reset_index(drop=True)
        
        if balance_total_tiles[i] == True:
            #Resample tiles to have equal number of class examples following previous balancing:        
            
            use_fn=bal_dat.loc[use_tiles,'fn']
            use_class=bal_dat.loc[use_tiles,'class']=='pos'
            npos=np.sum(use_class)
            nneg=len(use_class)-npos
            use_n = np.min([npos,nneg])
            print('Balancing %s tiles... %d neg %d pos, use %d' % (set_lab[i],nneg, npos,use_n))
            if npos > nneg:
                old_pos=np.argwhere(np.array(use_class)==True)
                neg=np.argwhere(np.array(use_class)==False)
                new=resample(old_pos,
                             replace=False,
                             n_samples=use_n,
                             random_state=seed)
                bal_tiles_idx=np.concatenate((neg,new))
            elif nneg > npos:
                old_neg=np.argwhere(np.array(use_class)==True)
                pos=np.argwhere(np.array(use_class)==False)
                new=resample(old_neg,
                             replace=False,
                             n_samples=use_n,
                             random_state=seed)
                bal_tiles_idx=np.concatenate((new,pos))
            temp=pd.DataFrame(bal_dat.loc[bal_tiles_idx[:,0],('fn','slide','class')]).reset_index(drop=True)
        else:
            use_class=bal_dat.loc[:,'class']=='pos'
            npos=np.sum(use_class)
            nneg=len(use_class)-npos
            print('Not balancing %s ... %d neg %d pos, use all.' % (set_lab[i],nneg, npos))
            temp=pd.DataFrame(bal_dat.loc[:,('fn','slide','class')]).reset_index(drop=True)
        temp['is_valid']=i
        if i == 0:
            out=temp
        else:
            out=pd.concat((out,temp),axis=0)
    
    full_path_fn=[path.joinpath(use).joinpath(fn) for fn in out['fn']]
    out['full_path']=full_path_fn
    print('Save to %s' % csv_path.joinpath('train_valid_fold_%d.csv' % fold))
    out.to_csv(csv_path.joinpath('train_valid_fold_%d.csv' % fold))
    fold+=1
    


Fold 0
Train slides n=16 pos, 15 neg, Test slides n=0 pos, 4 neg
Balancing train tiles... 3568 neg 9339 pos, use 3568
Not balancing valid ... 1252 neg 0 pos, use all.
Save to /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/tiles_model/resnet18_true_10kfold/csv/train_valid_fold_0.csv

Fold 1
Train slides n=15 pos, 16 neg, Test slides n=1 pos, 3 neg
Balancing train tiles... 4100 neg 8282 pos, use 4100
Not balancing valid ... 600 neg 933 pos, use all.
Save to /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/tiles_model/resnet18_true_10kfold/csv/train_valid_fold_1.csv

Fold 2
Train slides n=14 pos, 17 neg, Test slides n=2 pos, 2 neg
Balancing train tiles... 4334 neg 8750 pos, use 4334
Not balancing valid ... 470 neg 681 pos, use all.
Save to /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/tiles_model/resnet18_true_10kfold/csv/train_valid_fold_2.csv

Fold 3
Train slides n=14 pos, 17 neg, Test slides n=

In [127]:
#Constrain GroupShuffleSplit so that there are only splits with equal n in classes for train:
k=2
splitter = GroupShuffleSplit(n_splits=1000, 
                             train_size=k,
                             random_state=seed)
ktrain=[]
ktest=[]
utrain=[]
des_splits=100
for slide_train_idx, slide_test_idx in splitter.split(slide_df.loc[:,'slide'],
                            slide_df.loc[:,'group'],                                         
                            groups=slide_df.loc[:,'slide']):
    if (np.sum(slide_df.loc[slide_train_idx,'class']==True) == k/2) \
        and (np.sum(slide_df.loc[slide_train_idx,'class']==False) == k/2):
        ktrain.append(slide_train_idx)
        ktest.append(slide_test_idx)
        utrain.append(str(np.sort(slide_train_idx)))
    if len(ktrain) == des_splits:
        break
print(len(ktrain),len(ktest),len(utrain))

100 100 100


In [118]:
# Understand GroupShuffleSplit uniquness across splits
seed=35
splitter = GroupShuffleSplit(n_splits=100, 
                             train_size=4,
                             random_state=seed)
keep=[]
# 
for slide_train_idx, slide_test_idx in splitter.split(slide_df.loc[:,'slide'],
                            slide_df.loc[:,'group'],                                         
                            groups=slide_df.loc[:,'slide']):
    sets=[slide_train_idx,slide_test_idx]
    keep.append(str(np.sort(slide_train_idx)))
    set_lab=['train','valid']
    df_list=[]
    n_train_test = [np.sum(slide_df.loc[slide_train_idx,'class']==True), #n pos
                    np.sum(slide_df.loc[slide_train_idx,'class']==False), #n neg
                    np.sum(slide_df.loc[slide_test_idx,'class']==True), #n pos
                    np.sum(slide_df.loc[slide_test_idx,'class']==False)] #n neg
    if (np.sum(slide_df.loc[slide_train_idx,'class']==True) == 0) \
        or (np.sum(slide_df.loc[slide_train_idx,'class']==False) == 0):
        print('Train slides n=%d pos, %d neg, Test slides n=%d pos, %d neg' % tuple(n_train_test))
    

print(len(np.unique(keep)))

Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
Train slides n=4 pos, 0 neg, Test slides n=12 pos, 19 neg
Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
Train slides n=0 pos, 4 neg, Test slides n=16 pos, 15 neg
100


In [85]:
# Create cross validation object
crossval_method = '5fold'
seed=42
n_repeats = 5
if 'fold' in crossval_method:
    k = int(crossval_method.split('f')[0])
    
if crossval_method == 'jackknife':        
    #Leave-one-out jack-knife    
    splitter=LeaveOneGroupOut()
elif crossval_method == '10fold':
    #Group 10kFold

    splitter = RepeatedKFold(n_splits=k, 
                             n_repeats=n_repeats,
                             random_state=seed)
elif crossval_method == '5fold':
    splitter = RepeatedKFold(n_splits=k, 
                             n_repeats=n_repeats,
                             random_state=seed)
    
balance_tiles_per_slide=[True, False]
ntarget=300
fold=0 #Initialize
all_test=[]
balance_classes_total_tiles=[True,False]
for slide_train_idx, slide_test_idx in splitter.split(slide_df.loc[:,'slide'],
                            slide_df.loc[:,'group'],                                         
                            groups=slide_df.loc[:,'slide']):
    print('\nFold %d' % fold)
    sets=[slide_train_idx,slide_test_idx]
    set_lab=['train','valid']
    df_list=[]
    n_train_test = [np.sum(slide_df.loc[slide_train_idx,'class']==True), #n pos
                    np.sum(slide_df.loc[slide_train_idx,'class']==False), #n neg
                    np.sum(slide_df.loc[slide_test_idx,'class']==True), #n pos
                    np.sum(slide_df.loc[slide_test_idx,'class']==False)] #n neg
    print('Train slides n=%d pos, %d neg, Test slides n=%d pos, %d neg' % tuple(n_train_test))
    for i,ds in enumerate(sets):   
        use_slides=list(slide_df.loc[ds,'slide'])

        #balance slides (?): i.e. only draw from equal numbers of top level data (slides)
        
        # Balance n tiles drawn per-slide:
        n=[]
        on=[]
        if balance_tiles_per_slide[i] == True:
            for ii,slide_id in enumerate(use_slides):
                keep = np.argwhere(np.array(all_dat.loc[:,'slide'].isin([slide_id]))).flatten()
                n0 = keep.shape[0]
                on.append(n0)
                if n0 > ntarget:
                    keep= resample(keep,
                          n_samples=ntarget,
                          random_state=fold,
                          replace = False,)
                if ii==0:
                    bal_dat = all_dat.loc[keep,('fn','slide','class')].reset_index(drop=True)
                else:
                    temp=all_dat.loc[keep,('fn','slide','class')].reset_index(drop=True)
                    bal_dat= pd.concat((bal_dat,temp),axis=0)
                    
                ntiles=keep.shape[0]
                n.append(ntiles)
            print('\t pre rebalance: median n tiles / svs %s = %4.1f' % (set_lab[i],np.median(on)))
            print('\t after rebalance: median n tiles / svs %s = %4.1f' % (set_lab[i],np.median(n)))
            bal_dat.reset_index(inplace=True,drop=True)
        else:
            #Initialize dataframe with all tiles from use_slides:
            use_tiles=all_dat.loc[:,'slide'].isin(use_slides) #Index of tiles from slide(s) in current set 'ds' (train / valid)
            bal_dat=pd.DataFrame(all_dat.loc[use_tiles,('fn','slide','class')]).reset_index(drop=True)
        
        # Balance final classes from tiles in bal_dat:
        if balance_classes_total_tiles[i] == True:
            #Resample tiles to have equal number of class examples following any previous balancing:  
            use_fn=bal_dat.loc[:,'fn']
            use_class=bal_dat.loc[:,'class']=='pos'
            npos=np.sum(use_class)
            nneg=len(use_class)-npos
            use_n = np.min([npos,nneg])
            print('Balancing %s tiles... %d neg %d pos, use %d' % (set_lab[i],nneg, npos,use_n))
            if npos > nneg:
                old_pos=np.argwhere(np.array(use_class)==True)
                neg=np.argwhere(np.array(use_class)==False)
                new=resample(old_pos,
                             replace=False,
                             n_samples=use_n,
                             random_state=fold)
                bal_tiles_idx=np.concatenate((neg,new))
            elif nneg > npos:
                old_neg=np.argwhere(np.array(use_class)==False)
                pos=np.argwhere(np.array(use_class)==True)
                new=resample(old_neg,
                             replace=False,
                             n_samples=use_n,
                             random_state=fold)
                bal_tiles_idx=np.concatenate((new,pos))
            temp=pd.DataFrame(bal_dat.loc[bal_tiles_idx[:,0],('fn','slide','class')]).reset_index(drop=True)
        else:
            use_class=bal_dat.loc[:,'class']=='pos'
            npos=np.sum(use_class)
            nneg=len(use_class)-npos
            print('Not balancing %s ... %d neg %d pos, use all.' % (set_lab[i],nneg, npos))
            temp=pd.DataFrame(bal_dat.loc[:,('fn','slide','class')]).reset_index(drop=True)
        temp['is_valid']=i
        if i == 0:
            out=temp
        else:
            out=pd.concat((out,temp),axis=0)
    print('n_pos =',
          np.sum(np.array(out.loc[out.loc[:,'is_valid']==0,'class'])=='pos'), 
          '(should ==) n_neg=', 
          np.sum(np.array(out.loc[out.loc[:,'is_valid']==0,'class'])=='neg')
         )    
    fold += 1



Fold 0
Train slides n=12 pos, 16 neg, Test slides n=4 pos, 3 neg
	 pre rebalance: median n tiles / svs train = 946.0
	 after rebalance: median n tiles / svs train = 300.0
Balancing train tiles... 4149 neg 3600 pos, use 3600
Not balancing valid ... 1580 neg 5963 pos, use all.
n_pos = 3600 (should ==) n_neg= 3600

Fold 1
Train slides n=13 pos, 15 neg, Test slides n=3 pos, 4 neg
	 pre rebalance: median n tiles / svs train = 1003.0
	 after rebalance: median n tiles / svs train = 300.0
Balancing train tiles... 4090 neg 3830 pos, use 3830
Not balancing valid ... 4339 neg 4745 pos, use all.
n_pos = 3830 (should ==) n_neg= 3830

Fold 2
Train slides n=13 pos, 15 neg, Test slides n=3 pos, 4 neg
	 pre rebalance: median n tiles / svs train = 790.5
	 after rebalance: median n tiles / svs train = 300.0
Balancing train tiles... 4088 neg 3830 pos, use 3830
Not balancing valid ... 3550 neg 14783 pos, use all.
n_pos = 3830 (should ==) n_neg= 3830

Fold 3
Train slides n=13 pos, 15 neg, Test slides n=3 p

In [82]:
print(len(bal_tiles_idx),bal_dat.shape[0], len(np.unique(bal_tiles_idx)),max(bal_tiles_idx[:,0]))
print(np.sum(out.loc[:,'is_valid']),out.shape[0])
print('n_pos',np.sum(np.array(out.loc[out.loc[:,'is_valid']==False,'class'])=='pos'), 'should == n_neg', 
     np.sum(np.array(out.loc[out.loc[:,'is_valid']==False,'class'])=='neg'),)

7266 7857 7266 7762
7857 15123
n_pos 3633 should == n_neg 3633


In [52]:
fold = 0
csv_path=Path('~/data/biliseq_proc/v6/tiles_model/resnet18_jackknife_v0/csv')
df=pd.read_csv(csv_path.joinpath('train_valid_fold_%d.csv' % fold))
tissue =DataBlock(blocks=(ImageBlock, CategoryBlock),
                  get_x=ColReader('full_path'),
                  splitter=ColSplitter('is_valid') ,# GrandparentSplitter(train_name='train',valid_name='valid'),
                  get_y=  ColReader('class'), #using_attr(RegexLabeller(re_class), 'name'),
                  item_tfms=Resize(460) if use == 'summary' else None,
                  batch_tfms=aug_transforms(size=224,
                                            max_rotate=45,
                                            min_scale=0.75,
                                            flip_vert=True,
                                           )
                             ) 
dls = tissue.dataloaders(df, bs = 128)
learn = cnn_learner(dls, eval('resnet34'),
                metrics=[error_rate, accuracy],
                ).to_fp16()

/ihome/rbao/bri8/envs/fastai/lib/python3.8/site-packages/torch/_tensor.py:1023: UserWarning: torch.solve is deprecated in favor of torch.linalg.solveand will be removed in a future PyTorch release.
torch.linalg.solve has its arguments reversed and does not return the LU factorization.
To get the LU factorization see torch.lu, which can be used with torch.lu_solve or torch.lu_unpack.
X = torch.solve(B, A).solution
should be replaced with
X = torch.linalg.solve(A, B) (Triggered internally at  /opt/conda/conda-bld/pytorch_1631630839582/work/aten/src/ATen/native/BatchLinearAlgebra.cpp:760.)
  ret = func(*args, **kwargs)


In [16]:
for fold in range(0,1):
    print('Fold :', fold )
    df=pd.read_csv(csv_path.joinpath('train_valid_fold_%d.csv' % fold))
    tissue =DataBlock(blocks=(ImageBlock, CategoryBlock),
                      get_x=ColReader('full_path'),
                      splitter=ColSplitter('is_valid') ,# GrandparentSplitter(train_name='train',valid_name='valid'),
                      get_y=  ColReader('class'), #using_attr(RegexLabeller(re_class), 'name'),
                      item_tfms=Resize(460) if use == 'summary' else None,
                      batch_tfms=aug_transforms(size=224,
                                                max_rotate=45,
                                                min_scale=0.75,
                                                flip_vert=True,
                                               )
                                 ) 
    dls = tissue.dataloaders(df, bs = 128)
    learn = cnn_learner(dls, resnet34,
                    metrics=[error_rate, accuracy],
                    ).to_fp16()

    learn.fine_tune(5,freeze_epochs=1)
    learn.export(fname=export_path.joinpath('resnet18_1_5_%d.pkl' % fold))
    print('Performing inference.')
    #Infer:
    pred=learn.get_preds(with_decoded=True)

Fold : 0


epoch,train_loss,valid_loss,error_rate,accuracy,time
0,0.753875,1.123065,0.679755,0.320245,06:16


epoch,train_loss,valid_loss,error_rate,accuracy,time
0,0.456633,1.440734,0.725153,0.274847,06:36
1,0.314905,1.212813,0.557055,0.442945,06:37
2,0.231111,1.378717,0.522699,0.477301,06:39
3,0.178655,1.567806,0.585276,0.414724,06:41
4,0.154548,1.785606,0.619632,0.380368,06:36


Performing inference.


In [10]:
pred=learn.get_preds(with_decoded=True)

In [17]:
print(pred[1][0:10], pred[2][0:10])
correct = np.array(pred[1]) == np.array(pred[2])
print(np.sum(correct)/len(correct))

TensorCategory([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]) tensor([1, 1, 1, 0, 1, 1, 0, 1, 0, 1])
0.3803680981595092


In [46]:
df=pd.read_csv(csv_path.joinpath('train_valid_fold_%d.csv' % 0))

In [54]:
test_df=df.loc[df.is_valid==True,('slide','class')].reset_index(drop=True)
slides_c= ['%s_%s' slide,c for slide,c in test_df]
# u_slides_c=np.unique(slides_c)
# temp = [item.split('_') for item in u_slides_c]
# fold_df=pd.DataFrame(temp,columns=['slide','class'])
fold_df=pd.DataFrame((test_df))
test_slides=np.unique(test_df.loc[:,'slide'])
p=np.array(pred[0]) # Probability ['neg','pos'] (Can check with dls.vocab )
c=np.array(pred[2]) #Predictions decoded
fold_df['p1_fold_%d' % fold] = np.zeros((fold_df.shape[0],1))*np.nan
fold_df['wta_fold_%d' % fold] = np.zeros((fold_df.shape[0],1))*np.nan

for slide in test_slides:
    src_idx = np.array(test_df.loc[:,'slide']) == slide
    p1 = np.mean(p[src_idx,1])
    if np.sum(c[src_idx]==0) > np.sum(c[src_idx]==1):
        wta=0
    else:
        wta=1
    dest_idx = np.array(fold_df.loc[:,'slide'])== str(slide)
    # print(dest_idx,p1)
    fold_df.loc[dest_idx,'p1_fold_%d' % fold] = p1
    fold_df.loc[dest_idx,'wta_fold_%d' % fold] = wta
fold_df.head()

<ipython-input-54-7420389bdeb0>:19: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  dest_idx = np.array(fold_df.loc[:,'slide'])== str(slide)


KeyError: 'cannot use a single bool to index into setitem'

ValueError: too many values to unpack (expected 3)